In [ ]:
import  jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable





In [ ]:
## Parameters
x_lb, x_rb = 0, 1
n =4 
N_x = 2**n
L = x_rb - x_lb
dx = L/N_x
dt = 1e-3
steps=100
## 3D Grid
xs, ys, zs = jnp.meshgrid(
    jnp.linspace(x_lb, x_rb, N_x, endpoint=False), 
    jnp.linspace(x_lb, x_rb, N_x, endpoint=False), 
    jnp.linspace(x_lb, x_rb, N_x, endpoint=False),
    indexing='ij'
)

## Diffusion Coefficient Matrix A, 
# div(A grad u) = f
A = jnp.array([
    [3.0, 1.0, 0.0], 
    [1.0, 2.0, 1.0],
    [0.0, 1.0, 3.0]
])
#A = jnp.eye(3)
    


def spectral_eigenvalues(N, L=1.0):
    """Eigenvalues of the 1D derivative operator with periodic condition."""
    # 2j * pi * k
    k = jnp.fft.fftfreq(N, d=L/N) * 2j * jnp.pi  
    return k

In [ ]:
import jax

#@jax.vmap
#@jax.vmap
#@jax.vmap
#def A_vec(u):
#    return A@ u

#@jax.vmap
#@jax.vmap
#@jax.vmap
#def vec_dot(u, v):
#    return jnp.dot(u, v)

#def energy(u, f,N):
#
 #   u_h = np.fft.fftn(u)
#    wave = jnp.fft.fftfreq(N, d= dx) * 2j* np.pi
#    k_x, k_y, k_z  = np.meshgrid(wave, wave, wave, indexing='ij')
#    u_x_h = k_x * u_h
#    u_y_h = k_y * u_h
#    u_z_h = k_z * u_h
#    u_vec_h = np.stack([u_x_h, u_y_h, u_z_h], axis=-1)
#    Au_vec = A_vec(u_vec_h)
#
#    return jnp.mean(vec_dot(Au_vec, u_vec_h)/2 + f*u).real
@jax.vmap
@jax.vmap
@jax.vmap
def A_vec(u):
    return A@ u

@jax.vmap
@jax.vmap
@jax.vmap
def vec_dot(u, v):
    return jnp.dot(u, v)

def energy(u,f,N):
    u_h = np.fft.fftn(u)
    wave = np.fft.fftfreq(N_x, d= dx) * 2j* np.pi
    k_x, k_y, k_z  = np.meshgrid(wave, wave, wave, indexing='ij')
    u_x_h = k_x * u_h
    u_y_h = k_y * u_h
    u_z_h = k_z * u_h
    u_x = jnp.fft.ifft(u_x_h)
    u_y = jnp.fft.ifft(u_y_h)
    u_z = jnp.fft.ifft(u_z_h)
    u_grad = jnp.stack([u_x, u_y, u_z], axis=-1)
    Au_vec = A_vec(u_grad)

    return np.mean(vec_dot(Au_vec, u_grad)/2 + f*u).real

In [ ]:
def f(x, y, z) : 
    return np.cos(2*np.pi *x) * np.sin(-4*np.pi * y) * np.cos(2*np.pi*z)
    

def u_init(x, y, z): 
    return np.cos(2*np.pi *x) * np.sin(8*np.pi *y) + 2 * np.sin(6*np.pi*y) + 3 * np.sin(10*np.pi *x) * np.cos(12 * np.pi *y)**2 + np.sin(2*np.pi*z)

In [ ]:

"""Constructs the 1D DFT matrix of size N."""
dfmtx = np.fft.fft(np.eye(N_x))#/np.sqrt(N_x)

"""Constructs the 2D DFT matrix of size N x N as a Kronecker product."""
FG = np.kron(dfmtx, dfmtx)
GF = np.kron(
    (np.conj(dfmtx).T)/N_x, 
    (np.conj(dfmtx).T)/N_x, 
)

In [ ]:
def iterative_heat(u_n, f, dt=dt): 

    u_h = np.fft.fftn(u_n)
    f_h = np.fft.fftn(f)

    RHS =  u_h - dt* f_h

    # Invert (\Id - dt \Delta)    
    wave = np.fft.fftfreq(N_x, d= dx) * 2j* np.pi
    k_x, k_y, k_z = np.meshgrid(wave, wave, wave, indexing='ij')
    diffuse_u_h = (A[0,0]*(k_x**2) + A[1, 1]*(k_y**2) + A[2, 2]*(k_z**2)
                   + 2*(A[0, 1] * k_x * k_y + A[1, 2] * k_y * k_z + A[2,0]*k_z *k_x))

    ones = np.ones_like(diffuse_u_h)
    u_next_h = RHS/(ones - dt* diffuse_u_h)
    
    return  np.fft.ifftn(u_next_h)

In [ ]:

def spectralFilter(N, dt):
    d = spectral_eigenvalues(N)
    D= np.diag(d)
    eye = np.eye(N)
    Diffuse_h = (A[0,0]*np.kron(np.kron(D**2, eye), eye) + A[1, 1]*np.kron(eye, np.kron(D**2, eye)) + A[2, 2] *np.kron(eye, np.kron(eye, D**2)))
    Diffuse_h += 2*(A[0, 1]* np.kron(D, np.kron(D, eye)) + A[1, 2]* np.kron(eye, np.kron(D, D)) + A[2, 0]*np.kron(D, np.kron(eye, D)))
    op_h = np.kron(np.kron(eye, eye), eye) - dt *Diffuse_h 
    #print("Condition number of D:", np.linalg.cond(op_h))
    #m_h = np.diag( 1.0/np.diag(op_h))
    #return m_h 

    return np.linalg.inv(op_h)

def iterative_heat_FG(u_val, f_val, dt): 
    N = u_val.shape[0]
    # 1D DFT matrix
    dfmtx = np.fft.fft(np.eye(N))

    # 3D DFT Matrix via Kronecker: F x F x F
    FG = np.kron(np.kron(dfmtx, dfmtx), dfmtx)

    # Inverse 3D DFT Matrix
    dfmtx_inv = np.conj(dfmtx).T / N
    GF = np.kron(np.kron(dfmtx_inv, dfmtx_inv), dfmtx_inv)

    u_flatten = u_val.flatten()
    f_flatten = f_val.flatten()

    u_h = FG @ u_flatten
    f_h = FG @ f_flatten

    u_new = GF @ spectralFilter(N, dt) @ (u_h - dt * f_h)

    return u_new.reshape(N_x, N_x, N_x)



In [ ]:
import numpy as np
from scipy.linalg import sqrtm, dft
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
def make_unitary(A):
    N = A.shape[0]
    sqrt_term = sqrtm(np.eye(N) - A.conj().T @ A)
    return np.block([
        [A,         sqrt_term],
        [sqrt_term, -A       ]
    ])




# ── Quantum circuit ────────────────────────────────────────────────────────────

def build_circuit(inverse_Elliptic, n):

    total_qubits = 3 * n + 1
    # Normalise so that A / alpha has spectral norm ≤ 1
    alpha    = np.linalg.norm(inverse_Elliptic)
    A_normed = inverse_Elliptic / alpha

    U = make_unitary(A_normed)
  
    qft  = QFT(n)
    qft1 = QFT(n)
    qft2 = QFT(n)

    qc = QuantumCircuit(total_qubits)
    qc.append(qft,          list(range(n)))
    qc.append(qft1,         list(range(n, 2 * n)))
    qc.append(qft2,         list(range(2 * n, 3 * n)))
    qc.unitary(U,           list(range(total_qubits)))
    qc.append(qft.inverse(),  list(range(n)))
    qc.append(qft1.inverse(), list(range(n, 2 * n)))
    qc.append(qft2.inverse(), list(range(2 * n, 3 * n)))

    return qc, alpha


def extract_unitary(qc):
    """Simulates the circuit and returns its full unitary matrix."""
    print("  Simulating unitary...")
    qc_sim = qc.copy()
    qc_sim.save_unitary()
    sim    = AerSimulator(method="unitary")
    qc_sim = transpile(qc_sim, sim)
    result = sim.run(qc_sim).result()
    return np.array(result.get_unitary(qc_sim))



In [ ]:

def iterative_heat_QC(u_n, f, dt=dt): 
    v_n = u_n -dt*f
    v_n = v_n.reshape(N_x**3, 1)
    normalized_input = (v_n) / np.linalg.norm(v_n)
    #print(normalized_input.shape, mat[:N**2, :N**2].shape)
    u_next = leading_block @ (alpha * normalized_input)*np.linalg.norm(v_n)
    return  u_next.reshape(N_x, N_x,N_x)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors


def plot_solutions_and_errors(xs, ys, zs, u_theo, u_spec, u_quantum, label='', save_path=None):
    """
    3D Voxel visualization for Elliptic solutions.
    Unified 'turbo' colormap, high-vibrancy alpha, and fixed isometric rotation.
    """
    # 1. Prepare data
    u_theo_r    = np.array(u_theo).real
    u_spec_r    = np.array(u_spec).real
    u_quantum_r = np.array(u_quantum).real

    # Side length 8 for 512 total points
    N_side = int(np.round(u_theo_r.size**(1/3)))
    
    # 2. Reshape to 3D cube (8, 8, 8)
    # Using 'ij' meshgrid logic, the reshape preserves the spatial orientation
    u_theo_3d    = u_theo_r.reshape(N_side, N_side, N_side)
    u_spec_3d    = u_spec_r.reshape(N_side, N_side, N_side)
    u_quantum_3d = u_quantum_r.reshape(N_side, N_side, N_side)

    err_FG = np.abs(u_spec_3d - u_theo_3d)
    err_QC = np.abs(u_quantum_3d - u_theo_3d)

    fig = plt.figure(figsize=(20, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.1, wspace=0.2)
    
    # Scaling limits
    vmin_u, vmax_u = u_theo_3d.min(), u_theo_3d.max()
    vmax_e = max(err_FG.max(), err_QC.max()) or 1e-10

    def render_voxels(ax, data_3d, vmin, vmax, title):
        # Full 8x8x8 dense grid
        filled = np.ones(data_3d.shape, dtype=bool)
        
        # Color mapping: Turbo for high brightness and contrast
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        cmap = plt.get_cmap('turbo') 
        
        facecolors = cmap(norm(data_3d))
        facecolors[..., 3] = 0.65  # Vivid alpha for transparency
        
        # Plotting the voxels with thin white borders for "glow"
        ax.voxels(filled, facecolors=facecolors, edgecolors='white', 
                  linewidth=0.08, shade=True)
        
        ax.set_title(title, fontsize=15, pad=15, fontweight='bold')
        
        # --- ROTATION SETTINGS ---
        # elev: vertical angle, azim: horizontal rotation
        # -45 azim puts the (0,0) origin at the bottom-front
        ax.view_init(elev=25, azim=-45)
        
        ax.set_axis_off() 
        ax.set_box_aspect([1, 1, 1]) 
        
        return plt.cm.ScalarMappable(norm=norm, cmap=cmap)

    # --- Row 1: Solutions ---
    ax1 = fig.add_subplot(gs[0, 0], projection='3d')
    render_voxels(ax1, u_theo_3d, vmin_u, vmax_u, 'Reference')

    ax2 = fig.add_subplot(gs[0, 1], projection='3d')
    render_voxels(ax2, u_spec_3d, vmin_u, vmax_u, 'Classical (FG)')

    ax3 = fig.add_subplot(gs[0, 2], projection='3d')
    sm_u = render_voxels(ax3, u_quantum_3d, vmin_u, vmax_u, 'Quantum')
    
    # Solution Colorbar
    cax_u = fig.add_axes([0.92, 0.55, 0.015, 0.35])
    cb_u = fig.colorbar(sm_u, cax=cax_u)
    cb_u.set_label('Amplitude ($u$)', fontsize=12, fontweight='bold')

    # --- Row 2: Errors ---
    ax5 = fig.add_subplot(gs[1, 1], projection='3d')
    render_voxels(ax5, err_FG, 0, vmax_e, 'Classical Error')

    ax6 = fig.add_subplot(gs[1, 2], projection='3d')
    sm_e = render_voxels(ax6, err_QC, 0, vmax_e, 'Quantum Error')

    # Error Colorbar
    cax_e = fig.add_axes([0.92, 0.1, 0.015, 0.35])
    cb_e = fig.colorbar(sm_e, cax=cax_e)
    cb_e.set_label('Absolute Error', fontsize=12, fontweight='bold')

    if label:
        fig.suptitle(f"Parabolic Solver 3D Voxel Analysis: {label}", fontsize=22, y=0.98)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=200)
    plt.show()

In [ ]:

# Energy plot
def plot_energy(n, N, energy_classical, energy_FG, energy_quantum, steps, dt, save_path=None):
    ts = dt * np.arange(1, steps + 1)
    log_E_classical = np.log(np.array(energy_classical))
    log_E_FG = np.log(np.array(energy_FG))
    log_E_quantum = np.log(np.array(energy_quantum))

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(ts, log_E_classical, '-',  color='steelblue', lw=2, label=r'$E(u_{\mathrm{classical}}) - E_\infty$')
    ax.plot(ts, log_E_FG, '--', color='darkorange', lw=2, label=r'$E(u_{\mathrm{FG}}) - E_\infty$')
    ax.plot(ts, log_E_quantum, ':',  color='seagreen', lw=2.5, label=r'$E(u_{\mathrm{quantum}}) - E_\infty$')
    
    ax.set_xlabel('Time')
    ax.set_ylabel(r'$\log(E - E_\infty)$')
    ax.set_title(f'Energy decay  --  n = {n}, N = {N}x{N}')
    ax.legend()
    ax.grid(True, ls=':')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


In [ ]:
def solver_Elliptic_3D_fft(f, A, N, dx): 
    # Evaluate function on grid
    f_values = f(xs, ys, zs)

    # 1. Forward 3D FFT
    f_h = np.fft.fftn(f_values)
    
    # 2. Construct Wavenumbers
    # Get 1D wavenumbers
    wave = spectral_eigenvalues(N, dx*N) 
    
    # Create 3D grids of wavenumbers (kx, ky, kz)
    k_x, k_y, k_z = np.meshgrid(wave, wave, wave, indexing='ij')
    
    # Stack for vectorized computation: Shape (3, N, N, N)
    K_vec = np.array([k_x, k_y, k_z])
    
    # 3. Compute the denominator: sum(A_ij * ki * kj)
    # Using einsum for efficient calculation of A_ij * k_i * k_j
    denom = np.einsum('mn, m..., n... -> ...', A, K_vec, K_vec)
    
    # 4. Inversion (Handle Zero Mode)
    # Avoid division by zero at k=(0,0,0)
    denom[0, 0, 0] = 1.0
    u_h = f_h / denom
    
    # 5. Inverse 3D FFT
    return np.real(np.fft.ifftn(u_h))

In [ ]:
import os
import h5py
import numpy as np

# ── Updated Output Directory ──────────────────────────────────────────────────
out_dir = "result_diffusion_A_configs3D"
os.makedirs(out_dir, exist_ok=True)

A_configs = [
        {
        'label': 'Identity_3x3', 
        'matrix': jnp.eye(3)
    },
    {
        'label': 'Custom_3_1_05_3x3', 
        'matrix': np.array([
            [3.0, 1.0, 0.5],
            [1.0, 3.0, 1.0],
            [0.5, 1.0, 3.0]
        ])
    },
    {
        'label': 'Custom_10_1_1_3x3', 
        'matrix': np.array([
            [10.0, 0.0, 0.0],
            [0.0,  1.0, 0.0],
            [0.0,  0.0, 1.0]
        ])
    },
    {
        'label': 'Custom_1_100_1_3x3', 
        'matrix': np.array([
            [1.0, 0.0,   0.0],
            [0.0, 100.0, 0.0],
            [0.0, 0.0,   1.0]
        ])
    },
    {
        'label': 'Custom_1_100_01_3x3', 
        'matrix': np.array([
            [1.0, 0.0,   0.0],
            [0.0, 100.0, 0.0],
            [0.0, 0.0,   0.1]
        ])
    },
    {
        'label': 'Custom_1_1_100000_3x3', 
        'matrix': np.array([
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 100000.0]
        ])
    },
]
plt.rcParams.update({'font.size': 22})

for cfg in A_configs:
    label = cfg['label']
    print(f"\n{'='*55}")
    print(f"  A configuration : {label}")
    print(f"  Matrix:\n{cfg['matrix']}")
    print(f"{'='*55}")

    # ── Update global A ──
    A = cfg['matrix']

  #  @jax.vmap
   # @jax.vmap
    #def A_vec(u):
    #    return A @ u

    # ── Build inverse elliptic operator ──
    print("\n  Building inverse elliptic operator...")
    inverse_Elliptic = spectralFilter(N_x,dt)
    cond_number = float(np.linalg.cond(inverse_Elliptic))
    print(f"  Condition number of inverse elliptic operator: {cond_number:.2e}")
    # ── Build quantum circuit ──
    print("\n  Building quantum circuit...")
    qc, alpha = build_circuit(inverse_Elliptic, n)
    mat = extract_unitary(qc)

    # ── Time stepping ──
    print("\n  Time stepping...")
    block_size    = 2 ** (3 * n)
    leading_block = mat[:block_size, :block_size]

    u_theo    =  u_init(xs, ys, zs)
    u_spec    =  u_init(xs, ys, zs)
    u_quantum =  u_init(xs, ys, zs)
 

    energy_spec    = []
    energy_FG      = []
    energy_quantum = []

    # Reference Calculations
    u_ref  = solver_Elliptic_3D_fft(f, A, N_x, dx)
    u_ref  = u_ref.real
    E_ref  = energy(u_ref, f(xs, ys, zs), N_x)
    f_vals = f(xs, ys,zs)
  


    for _ in range(steps):
        u_theo    = iterative_heat(u_theo, f_vals,  dt=dt)
        u_spec    = iterative_heat_FG(u_spec, f_vals, dt=dt).real
        u_quantum = iterative_heat_QC(u_quantum, f_vals, dt=dt)
        
        # Energy calculations (relative to E_ref)
        energy_spec.append(energy(u_spec, f_vals, N_x) - E_ref + 1e-6)
        energy_FG.append(energy(u_spec, f_vals, N_x) - E_ref + 1e-6)
        energy_quantum.append(energy(u_quantum.real, f_vals, N_x) - E_ref + 1e-6)

    u_quantum       = u_quantum.reshape(N_x, N_x, N_x)
    error_classical = float(np.linalg.norm(u_spec - u_theo) / np.linalg.norm(u_theo))
    error_quantum   = float(np.linalg.norm(u_quantum - u_theo) / np.linalg.norm(u_theo))

    print(f"\n  {'─'*43}")
    print(f"  Classical error : {error_classical:.6e}")
    print(f"  Quantum error   : {error_quantum:.6e}")

    # ── Save results to HDF5 file ─────────────────────────────────────
    h5_path = os.path.join(out_dir, f"{label}_results.h5")
    with h5py.File(h5_path, 'w') as hf:
        # Final Iterative Solutions
        hf.create_dataset('u_theoretical', data=np.array(u_theo).real)
        hf.create_dataset('u_classical',   data=np.array(u_spec).real)
        hf.create_dataset('u_quantum',     data=np.array(u_quantum).real)
        
        # Reference Data
        hf.create_dataset('u_ref',         data=np.array(u_ref).real)        
        # Energy Trajectories
        hf.create_dataset('energy_spec',    data=np.array(energy_spec))
        hf.create_dataset('energy_FG',      data=np.array(energy_FG))
        hf.create_dataset('energy_quantum', data=np.array(energy_quantum))
        
        # Matrix and Metadata
        hf.create_dataset('matrix_A',      data=cfg['matrix'])
        hf.attrs['E_ref']            = E_ref
        hf.attrs['condition_number'] = cond_number
        hf.attrs['error_classical']  = error_classical
        hf.attrs['error_quantum']    = error_quantum
        hf.attrs['label']            = label
        
    print(f"  H5 Data saved:  {h5_path}")

    # ── Save Figures ──
    # path_sol = os.path.join(out_dir, f"{label}_solutions_errors.png")
    # plot_solutions_and_errors(n, N_x, xs, ys, u_theo, u_spec, u_quantum.real,
    #                           steps, dt, save_path=path_sol)
    path_fig = os.path.join(out_dir, f"{label}_solutions_errors.png")
    plot_solutions_and_errors(xs, ys, zs, u_theo, u_spec, u_quantum,
                              label=label, save_path=path_fig)
    print(f"  Figure saved : {path_fig}")

    path_energy = os.path.join(out_dir, f"{label}_energy.png")
    plot_energy(n, N_x, energy_spec, energy_FG, energy_quantum, steps, dt,
                save_path=path_energy)

print(f"\n{'='*55}")
print(f"  All done! Results saved in: '{out_dir}/'")
print(f"{'='*55}")